# 00 — Bronze: bids

Ingests the raw bid export exactly as it arrives. **No casting, no cleaning,
no filtering.** Anything that looks wrong here is preserved so the Silver
layer can decide what to do about it, and so the raw layer stays a faithful
copy of the source.

Schema drift is accepted rather than rejected: the source is a manual Excel
export whose columns change without notice, and failing the load on a
cosmetic change would stop the pipeline for no good reason. The data
contract is enforced at Silver.

**No cluster libraries required.** The Excel file is read with pandas +
openpyxl (already in `requirements.txt`) and converted to a Spark
DataFrame, instead of the `com.crealytics.spark.excel` Maven package —
that keeps this notebook runnable on any cluster with no setup step.

In [0]:
CATALOG = "bronze"
SCHEMA = "bid"
VOLUME_PATH = "/Volumes/raw/bid/bids/bids.xlsx"
TABLE = f"{CATALOG}.{SCHEMA}.bids"

spark.sql(f"CREATE CATALOG IF NOT EXISTS {CATALOG}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

## Read

`dtype=str` is pandas' equivalent of `inferSchema=false` — everything lands
as string, and Silver casts. Reading by sheet name only, never a fixed cell
range: a hardcoded range such as `Bronze!A1:K1583` silently truncates the
moment the source grows by one row, which is the worst class of bug — no
error, just missing data.

In [0]:
# %uv pip install openpyxl

In [0]:
import pandas as pd

pdf_raw = pd.read_excel(VOLUME_PATH, sheet_name="Bronze", dtype=str)
df_raw = spark.createDataFrame(pdf_raw)

print(f"rows: {df_raw.count()}   columns: {len(df_raw.columns)}")
df_raw.printSchema()

## Ingestion metadata

`_ingested_at` and `_source_file` make it possible to answer "when did this
row arrive and where did it come from" months later, without which incident
triage on a data pipeline is guesswork.

In [0]:
from pyspark.sql import functions as F

df_bronze = (
    df_raw
    .withColumn("_ingested_at", F.current_timestamp())
    .withColumn("_source_file", F.lit(VOLUME_PATH))
)

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .option("mergeSchema", "true")   # deliberate: see note below
    .saveAsTable(TABLE)
)

### On `mergeSchema`

Accepting schema evolution at Bronze is a choice, not a default. The upstream
export gains and loses columns without warning; refusing them would break
ingestion for a change that costs nothing to absorb. The cost is that a
renamed column arrives silently as a new one — which is why the check below
exists and why Silver validates against an explicit contract.

In [0]:
EXPECTED = {
    "bid_id", "created_at", "created_at_str", "is_confirmed_date", "bid_date",
    "closed_at", "closed_at_str", "outcome", "loss_reason", "competitor_name",
    "client_id", "contract_value_brl",
}

actual = set(df_bronze.columns) - {"_ingested_at", "_source_file"}
missing, unexpected = EXPECTED - actual, actual - EXPECTED

if missing:
    raise ValueError(f"Columns missing from source: {sorted(missing)}")
if unexpected:
    print(f"WARNING — new columns absorbed, review Silver: {sorted(unexpected)}")

print(f"Schema check passed. {spark.table(TABLE).count()} rows written to {TABLE}.")

In [0]:
%sql
select * from bronze.bid.bids